# VCF Zarr Benchmark: polars-bio vs sgkit vs zarr-python

This notebook benchmarks VCF Zarr reads and analytical queries across:

- **polars-bio** through `scan_vcf_zarr` and DataFusion target partitions
- **sgkit** through the installed sgkit/xarray/Dask stack when the selected VCZ store is compatible
- **zarr-python** through direct raw-array reads

The benchmark is end-to-end: it can download 1000 Genomes VCF inputs, convert them to VCF Zarr with bio2zarr/vcf2zarr, inspect the resulting store, run scenarios, and plot results. Expensive download and conversion steps are gated by environment variables and never run implicitly.

The tools expose different abstraction levels. polars-bio returns a logical VCF table, sgkit/xarray works with dataset variables, and zarr-python reads raw arrays. The notebook validates result shape/counts where possible and records unsupported scenarios as skipped rows.

## Configuration

Set environment variables before running the notebook:

```bash
export VCZ_BENCH_PROFILE=30x_chr22
export VCZ_BENCH_DATA_DIR=/path/to/vcz-bench-data
export VCZ_BENCH_ENABLE_DOWNLOAD=0
export VCZ_BENCH_ENABLE_CONVERT=0
export VCZ_BENCH_FORCE_REBUILD=0
export VCZ_BENCH_TARGET_PARTITIONS=1,2,4,8,16
export VCZ_BENCH_REPETITIONS=3
export VCZ_BENCH_WARMUPS=1
export VCZ_BENCH_SAMPLE_COUNT=16
export VCZ_BENCH_SAMPLES=
```

For a prebuilt store:

```bash
export VCZ_BENCH_PROFILE=custom
export VCZ_BENCH_CUSTOM_VCZ=/path/to/data.vcz
```

For custom VCF input conversion:

```bash
export VCZ_BENCH_PROFILE=custom
export VCZ_BENCH_CUSTOM_VCF=/path/to/input.vcf.gz
export VCZ_BENCH_CUSTOM_VCZ=/path/to/output.vcz
export VCZ_BENCH_ENABLE_CONVERT=1
```

In [ ]:
from __future__ import annotations

import importlib
import importlib.metadata as importlib_metadata
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable

import pandas as pd
import polars as pl
import polars_bio as pb

try:
    from IPython.display import display
except Exception:  # pragma: no cover - notebook fallback
    display = print

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

In [ ]:
def env_flag(name: str, default: bool = False) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


def env_int(name: str, default: int) -> int:
    value = os.environ.get(name)
    return default if value in (None, "") else int(value)


def env_csv(name: str, default: str = "") -> list[str]:
    value = os.environ.get(name, default)
    return [part.strip() for part in value.split(",") if part.strip()]


def package_version(name: str) -> str:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "not installed"


def optional_import(module_name: str):
    try:
        return importlib.import_module(module_name)
    except Exception as exc:
        print(f"optional dependency {module_name!r} unavailable: {exc}")
        return None


def parse_target_partitions() -> list[int]:
    values = env_csv("VCZ_BENCH_TARGET_PARTITIONS", "1,2,4,8,16")
    partitions = [int(value) for value in values]
    return sorted({max(1, value) for value in partitions})


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "polars_bio").exists():
            return candidate
    return current


REPO_ROOT = find_repo_root()


def resolve_path(value: str | Path) -> Path:
    path = Path(value).expanduser()
    return path if path.is_absolute() else REPO_ROOT / path


PROFILE_NAME = os.environ.get("VCZ_BENCH_PROFILE", "30x_chr22")
DATA_DIR = resolve_path(os.environ.get("VCZ_BENCH_DATA_DIR", REPO_ROOT / "tmp" / "vcz-bench-data"))
ENABLE_DOWNLOAD = env_flag("VCZ_BENCH_ENABLE_DOWNLOAD", False)
ENABLE_CONVERT = env_flag("VCZ_BENCH_ENABLE_CONVERT", False)
FORCE_REBUILD = env_flag("VCZ_BENCH_FORCE_REBUILD", False)
TARGET_PARTITIONS = parse_target_partitions()
REPETITIONS = env_int("VCZ_BENCH_REPETITIONS", 3)
WARMUPS = env_int("VCZ_BENCH_WARMUPS", 1)
SAMPLE_COUNT = env_int("VCZ_BENCH_SAMPLE_COUNT", 16)
REQUESTED_SAMPLES = env_csv("VCZ_BENCH_SAMPLES", "")
REQUESTED_REGION = os.environ.get("VCZ_BENCH_REGION", "").strip()

CONFIG = {
    "profile": PROFILE_NAME,
    "repo_root": str(REPO_ROOT),
    "data_dir": str(DATA_DIR),
    "enable_download": ENABLE_DOWNLOAD,
    "enable_convert": ENABLE_CONVERT,
    "force_rebuild": FORCE_REBUILD,
    "target_partitions": TARGET_PARTITIONS,
    "repetitions": REPETITIONS,
    "warmups": WARMUPS,
    "sample_count": SAMPLE_COUNT,
    "requested_samples": REQUESTED_SAMPLES,
    "requested_region": REQUESTED_REGION,
}

display(pd.DataFrame([CONFIG]).T.rename(columns={0: "value"}))

## Dataset Profiles

In [ ]:
FTP_BASE = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp"
PHASE3_RELEASE = f"{FTP_BASE}/release/20130502"
HIGH_COV_BASE = f"{FTP_BASE}/data_collections/1000G_2504_high_coverage/working/20220422_3202_phased_SNV_INDEL_SV"


def phase3_chr_url(chrom: int) -> str:
    return f"{PHASE3_RELEASE}/ALL.chr{chrom}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"


def high_cov_chr_url(chrom: int) -> str:
    return f"{HIGH_COV_BASE}/1kGP_high_coverage_Illumina.chr{chrom}.filtered.SNV_INDEL_SV_phased_panel.vcf.gz"


DATASET_PROFILES: dict[str, dict[str, Any]] = {
    "phase3_chr22": {
        "description": "1000 Genomes Phase 3 chr22 genotype VCF",
        "vcf_urls": [phase3_chr_url(22)],
        "index_urls": [f"{phase3_chr_url(22)}.tbi"],
        "default_region": "22:20000000-25000000",
        "notes": "Small fallback profile; chr22 VCF is about 196 MB compressed.",
    },
    "30x_chr22": {
        "description": "1000 Genomes 30x high-coverage chr22 phased SNV/INDEL/SV panel",
        "vcf_urls": [high_cov_chr_url(22)],
        "index_urls": [f"{high_cov_chr_url(22)}.tbi"],
        "default_region": "chr22:20000000-25000000",
        "notes": "Recommended default profile; chr22 VCF is about 425 MB compressed.",
    },
    "phase3_wgs_sites": {
        "description": "1000 Genomes Phase 3 WGS sites-only VCF",
        "vcf_urls": [f"{PHASE3_RELEASE}/ALL.wgs.phase3_shapeit2_mvncall_integrated_v5c.20130502.sites.vcf.gz"],
        "index_urls": [f"{PHASE3_RELEASE}/ALL.wgs.phase3_shapeit2_mvncall_integrated_v5c.20130502.sites.vcf.gz.tbi"],
        "default_region": "22:20000000-25000000",
        "notes": "Large variant-only profile; useful when genotype/sample work is not needed.",
    },
    "phase3_wgs_autosomes": {
        "description": "1000 Genomes Phase 3 chr1-22 genotype VCFs",
        "vcf_urls": [phase3_chr_url(chrom) for chrom in range(1, 23)],
        "index_urls": [f"{phase3_chr_url(chrom)}.tbi" for chrom in range(1, 23)],
        "default_region": "22:20000000-25000000",
        "notes": "Large multi-input profile using Phase 3 autosomes.",
    },
    "30x_wgs_autosomes": {
        "description": "1000 Genomes 30x high-coverage chr1-22 phased panel",
        "vcf_urls": [high_cov_chr_url(chrom) for chrom in range(1, 23)],
        "index_urls": [f"{high_cov_chr_url(chrom)}.tbi" for chrom in range(1, 23)],
        "default_region": "chr22:20000000-25000000",
        "notes": "Stress profile using 30x high-coverage autosomes.",
    },
    "legacy_wes_exome": {
        "description": "Older 1000 Genomes exome-like genotype VCF",
        "vcf_urls": [
            f"{FTP_BASE}/technical/working/20110414_broad_exome_vcf/ALL.BI.20110228.SNPs_and_indels.exome.genotypes.vcf.gz"
        ],
        "index_urls": [
            f"{FTP_BASE}/technical/working/20110414_broad_exome_vcf/ALL.BI.20110228.SNPs_and_indels.exome.genotypes.vcf.gz.tbi"
        ],
        "default_region": "22:20000000-25000000",
        "notes": "Legacy WES-style sparse/exome workload; older and less canonical than Phase 3/30x profiles.",
    },
    "custom": {
        "description": "User-supplied VCF or VCZ paths",
        "vcf_urls": [],
        "index_urls": [],
        "default_region": "",
        "notes": "Set VCZ_BENCH_CUSTOM_VCF and/or VCZ_BENCH_CUSTOM_VCZ.",
    },
}

if PROFILE_NAME not in DATASET_PROFILES:
    available = ", ".join(sorted(DATASET_PROFILES))
    raise ValueError(f"Unknown VCZ_BENCH_PROFILE={PROFILE_NAME!r}. Available profiles: {available}")

PROFILE = DATASET_PROFILES[PROFILE_NAME]
PROFILE_DIR = DATA_DIR / PROFILE_NAME
CUSTOM_VCF = env_csv("VCZ_BENCH_CUSTOM_VCF", "")
CUSTOM_VCZ = os.environ.get("VCZ_BENCH_CUSTOM_VCZ", "").strip()

if CUSTOM_VCF:
    VCF_PATHS = [resolve_path(path) for path in CUSTOM_VCF]
    INDEX_PATHS: list[Path] = []
else:
    VCF_PATHS = [PROFILE_DIR / Path(url).name for url in PROFILE["vcf_urls"]]
    INDEX_PATHS = [PROFILE_DIR / Path(url).name for url in PROFILE["index_urls"]]

VCZ_PATH = resolve_path(CUSTOM_VCZ) if CUSTOM_VCZ else PROFILE_DIR / f"{PROFILE_NAME}.vcz"

profile_rows = []
for name, profile in DATASET_PROFILES.items():
    profile_rows.append(
        {
            "profile": name,
            "description": profile["description"],
            "inputs": len(profile["vcf_urls"]),
            "default_region": profile.get("default_region", ""),
            "notes": profile.get("notes", ""),
        }
    )

display(pd.DataFrame(profile_rows))
print(f"Selected profile: {PROFILE_NAME}")
print(f"VCZ path: {VCZ_PATH}")

## Gated Data Preparation

In [ ]:
def download_file(url: str, output: Path, force: bool = False) -> None:
    if output.exists() and not force:
        print(f"reusing {output}")
        return
    output.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["curl", "-L", "--fail", "--continue-at", "-", "--output", str(output), url],
        check=True,
    )


def run_vcf2zarr(vcf_paths: list[Path], output_vcz: Path, force: bool = False) -> None:
    if output_vcz.exists() and not force:
        print(f"reusing {output_vcz}")
        return
    exe = shutil.which("vcf2zarr")
    cmd_prefix = [exe] if exe else [sys.executable, "-m", "bio2zarr", "vcf2zarr"]
    icf_path = output_vcz.with_suffix(".icf")
    if force:
        shutil.rmtree(output_vcz, ignore_errors=True)
        shutil.rmtree(icf_path, ignore_errors=True)
    output_vcz.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([*cmd_prefix, "explode", *map(str, vcf_paths), str(icf_path)], check=True)
    subprocess.run([*cmd_prefix, "encode", str(icf_path), str(output_vcz)], check=True)


PROFILE_DIR.mkdir(parents=True, exist_ok=True)
missing_vcf_paths = [path for path in VCF_PATHS if not path.exists()]

prep_rows = [
    {"kind": "vcf", "path": str(path), "exists": path.exists()} for path in VCF_PATHS
] + [
    {"kind": "index", "path": str(path), "exists": path.exists()} for path in INDEX_PATHS
] + [
    {"kind": "vcz", "path": str(VCZ_PATH), "exists": VCZ_PATH.exists()}
]
display(pd.DataFrame(prep_rows))

if missing_vcf_paths and not ENABLE_DOWNLOAD and not VCZ_PATH.exists():
    missing = "\n".join(str(path) for path in missing_vcf_paths[:5])
    raise RuntimeError(
        "Missing VCF inputs and VCZ output is not present. "
        "Set VCZ_BENCH_ENABLE_DOWNLOAD=1, provide VCZ_BENCH_CUSTOM_VCF, "
        f"or provide VCZ_BENCH_CUSTOM_VCZ. First missing paths:\n{missing}"
    )

if ENABLE_DOWNLOAD:
    for url, path in zip(PROFILE["vcf_urls"], VCF_PATHS):
        download_file(url, path, FORCE_REBUILD)
    for url, path in zip(PROFILE["index_urls"], INDEX_PATHS):
        download_file(url, path, FORCE_REBUILD)

if not VCZ_PATH.exists():
    if not ENABLE_CONVERT:
        raise RuntimeError(
            "Missing VCZ output. Set VCZ_BENCH_ENABLE_CONVERT=1 or provide VCZ_BENCH_CUSTOM_VCZ. "
            f"Expected VCZ path: {VCZ_PATH}"
        )
    if not VCF_PATHS:
        raise RuntimeError("No VCF inputs available for conversion. Set VCZ_BENCH_CUSTOM_VCF or choose a non-custom profile.")
    run_vcf2zarr(VCF_PATHS, VCZ_PATH, FORCE_REBUILD)

print(f"Ready to benchmark VCZ store: {VCZ_PATH}")

## Optional Dependencies And Store Metadata

In [ ]:
np = optional_import("numpy")
zarr = optional_import("zarr")
sgkit = optional_import("sgkit")
xarray = optional_import("xarray")
dask = optional_import("dask")
matplotlib = optional_import("matplotlib")
seaborn = optional_import("seaborn")


def format_bytes(path: Path) -> str:
    if path.is_file():
        size = path.stat().st_size
    elif path.is_dir():
        size = sum(child.stat().st_size for child in path.rglob("*") if child.is_file())
    else:
        return "missing"
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024 or unit == "TB":
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} TB"


def decode_scalar(value: Any) -> Any:
    if hasattr(value, "item"):
        try:
            value = value.item()
        except Exception:
            pass
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    return value


def decode_list(values: Any, limit: int | None = None) -> list[Any]:
    if values is None:
        return []
    if limit is not None:
        values = values[:limit]
    if hasattr(values, "tolist"):
        values = values.tolist()
    return [decode_scalar(value) for value in values]


def open_zarr_group(path: Path):
    if zarr is None:
        raise RuntimeError("zarr-python is not installed")
    return zarr.open_group(str(path), mode="r")


def zarr_array_keys(group) -> list[str]:
    try:
        return sorted(group.array_keys())
    except Exception:
        keys = []
        for key in group.keys():
            try:
                obj = group[key]
                if hasattr(obj, "shape"):
                    keys.append(key)
            except Exception:
                continue
        return sorted(keys)


try:
    ZARR_GROUP = open_zarr_group(VCZ_PATH)
    ARRAY_NAMES = zarr_array_keys(ZARR_GROUP)
except Exception as exc:
    print(f"zarr metadata unavailable: {exc}")
    ZARR_GROUP = None
    ARRAY_NAMES = []


def has_array(name: str) -> bool:
    return ZARR_GROUP is not None and name in ARRAY_NAMES


def zarr_array(name: str):
    if not has_array(name):
        raise SkipBenchmark(f"VCZ array {name!r} is not available")
    return ZARR_GROUP[name]


def read_zarr(name: str, selection: Any = slice(None)):
    return zarr_array(name)[selection]


def first_n(name: str, n: int = 10) -> list[Any]:
    if not has_array(name):
        return []
    arr = zarr_array(name)
    size = arr.shape[0] if arr.shape else 0
    return decode_list(arr[: min(size, n)])


def infer_variant_count() -> int | None:
    if has_array("variant_position"):
        return int(zarr_array("variant_position").shape[0])
    try:
        df = pb.scan_vcf_zarr(str(VCZ_PATH)).select(pl.len().alias("n")).collect()
        return int(df["n"][0])
    except Exception:
        return None


def infer_sample_ids(limit: int | None = None) -> list[str]:
    if not has_array("sample_id"):
        return []
    try:
        arr = zarr_array("sample_id")
        count = arr.shape[0] if arr.shape else 0
        return [str(value) for value in decode_list(arr[: count if limit is None else min(count, limit)])]
    except Exception:
        return []


def contig_lookup() -> dict[int, str]:
    if not has_array("contig_id"):
        return {}
    values = first_n("contig_id", zarr_array("contig_id").shape[0])
    return {idx: str(value) for idx, value in enumerate(values)}


def parse_region(region: str) -> tuple[str, int, int] | None:
    if not region:
        return None
    match = re.match(r"^([^:]+):(\d+)-(\d+)$", region.replace(",", ""))
    if not match:
        raise ValueError(f"Region must look like chrom:start-end, got {region!r}")
    chrom, start, end = match.groups()
    return chrom, int(start), int(end)


def infer_region() -> tuple[str, int, int]:
    explicit = parse_region(REQUESTED_REGION)
    if explicit is not None:
        return explicit
    profile_region = parse_region(PROFILE.get("default_region", ""))
    if PROFILE_NAME != "custom" and profile_region is not None:
        return profile_region
    if has_array("variant_position"):
        positions = read_zarr("variant_position", slice(0, min(1000, zarr_array("variant_position").shape[0])))
        pos_list = decode_list(positions)
        if pos_list:
            start = int(pos_list[0])
            end = start + max(10_000, min(1_000_000, max(pos_list) - start if len(pos_list) > 1 else 10_000))
            lookup = contig_lookup()
            chrom = ""
            if has_array("variant_contig"):
                contig_values = decode_list(read_zarr("variant_contig", slice(0, 1)))
                if contig_values:
                    chrom = lookup.get(int(contig_values[0]), str(contig_values[0]))
            return chrom or "", start, end
    df = pb.scan_vcf_zarr(str(VCZ_PATH)).select(["chrom", "start"]).head(1).collect()
    return str(df["chrom"][0]), int(df["start"][0]), int(df["start"][0]) + 10_000


def choose_info_field() -> str | None:
    for candidate in ["AF", "DP", "AC", "AN"]:
        if has_array(f"variant_{candidate}"):
            return candidate
    for name in ARRAY_NAMES:
        if name.startswith("variant_") and name not in {
            "variant_contig",
            "variant_position",
            "variant_length",
            "variant_allele",
            "variant_id",
            "variant_id_mask",
            "variant_filter",
            "variant_quality",
        }:
            return name.removeprefix("variant_")
    return None


def choose_format_field() -> str | None:
    for candidate in ["GT", "DP", "GQ"]:
        if has_array(f"call_{candidate}"):
            return candidate
    for name in ARRAY_NAMES:
        if name.startswith("call_"):
            return name.removeprefix("call_")
    return None


VARIANT_COUNT = infer_variant_count()
SAMPLE_IDS = infer_sample_ids()
SAMPLE_ID_TO_INDEX = {sample: idx for idx, sample in enumerate(SAMPLE_IDS)}
SAMPLES_FOR_QUERY = REQUESTED_SAMPLES or SAMPLE_IDS[:SAMPLE_COUNT]
BENCH_REGION = infer_region()
INFO_FIELD = choose_info_field()
FORMAT_FIELD = choose_format_field()

metadata = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "polars_bio": package_version("polars-bio"),
    "polars": package_version("polars"),
    "pandas": package_version("pandas"),
    "sgkit": package_version("sgkit"),
    "zarr": package_version("zarr"),
    "xarray": package_version("xarray"),
    "dask": package_version("dask"),
    "profile": PROFILE_NAME,
    "vcz_path": str(VCZ_PATH),
    "vcz_size": format_bytes(VCZ_PATH),
    "variant_count": VARIANT_COUNT,
    "sample_count": len(SAMPLE_IDS),
    "array_count": len(ARRAY_NAMES),
    "selected_region": f"{BENCH_REGION[0]}:{BENCH_REGION[1]}-{BENCH_REGION[2]}",
    "info_field": INFO_FIELD,
    "format_field": FORMAT_FIELD,
    "sample_query_count": len(SAMPLES_FOR_QUERY),
}

display(pd.DataFrame([metadata]).T.rename(columns={0: "value"}))
if ARRAY_NAMES:
    display(pd.DataFrame({"array": ARRAY_NAMES, "shape": [str(zarr_array(name).shape) for name in ARRAY_NAMES]}).head(80))

## Benchmark Harness

In [ ]:
class SkipBenchmark(Exception):
    pass


@dataclass
class ResultRow:
    tool: str
    scenario: str
    profile: str
    partition: str
    repetition: int
    status: str
    elapsed_s: float | None
    validation: str
    error: str
    notes: str


def validate_any(output: Any) -> str:
    if isinstance(output, pl.DataFrame):
        return f"rows={output.height}, cols={len(output.columns)}"
    if isinstance(output, pd.DataFrame):
        return f"rows={len(output)}, cols={len(output.columns)}"
    if isinstance(output, dict):
        parts = []
        for key in ["rows", "variants", "samples", "shape", "arrays", "columns"]:
            if key in output:
                parts.append(f"{key}={output[key]}")
        return ", ".join(parts) or "dict"
    if hasattr(output, "sizes"):
        return ", ".join(f"{k}={v}" for k, v in dict(output.sizes).items())
    if hasattr(output, "shape"):
        return f"shape={output.shape}"
    return type(output).__name__


def measure(
    tool: str,
    scenario: str,
    partition: str,
    fn: Callable[[], Any],
    validate: Callable[[Any], str] = validate_any,
    notes: str = "",
) -> list[ResultRow]:
    rows: list[ResultRow] = []
    try:
        for _ in range(WARMUPS):
            validate(fn())
        for rep in range(REPETITIONS):
            started = time.perf_counter()
            output = fn()
            elapsed = time.perf_counter() - started
            rows.append(
                ResultRow(
                    tool=tool,
                    scenario=scenario,
                    profile=PROFILE_NAME,
                    partition=partition,
                    repetition=rep,
                    status="ok",
                    elapsed_s=elapsed,
                    validation=validate(output),
                    error="",
                    notes=notes,
                )
            )
    except SkipBenchmark as exc:
        rows.append(ResultRow(tool, scenario, PROFILE_NAME, partition, -1, "skipped", None, "", str(exc), notes))
    except Exception as exc:
        rows.append(ResultRow(tool, scenario, PROFILE_NAME, partition, -1, "failed", None, "", repr(exc), notes))
    return rows


def core_columns() -> list[str]:
    return ["chrom", "start", "end", "ref", "alt"]


def available_core_columns(lf: pl.LazyFrame) -> list[str]:
    try:
        names = set(lf.collect_schema().names())
    except Exception:
        names = set(lf.columns)
    return [column for column in core_columns() if column in names]


def sampled_variants(limit: int = 5) -> list[tuple[str, int]]:
    if has_array("variant_position"):
        positions = [int(value) for value in decode_list(read_zarr("variant_position", slice(0, min(limit, zarr_array("variant_position").shape[0]))))]
        lookup = contig_lookup()
        chrom = BENCH_REGION[0]
        if has_array("variant_contig"):
            contigs = decode_list(read_zarr("variant_contig", slice(0, len(positions))))
            return [(lookup.get(int(contig), str(contig)), pos) for contig, pos in zip(contigs, positions)]
        return [(chrom, pos) for pos in positions]
    df = pb.scan_vcf_zarr(str(VCZ_PATH)).select(["chrom", "start"]).head(limit).collect()
    return [(str(chrom), int(pos)) for chrom, pos in zip(df["chrom"].to_list(), df["start"].to_list())]


LOOKUP_VARIANTS = sampled_variants()
LOOKUP_VARIANT = LOOKUP_VARIANTS[0] if LOOKUP_VARIANTS else BENCH_REGION[:2]
print(f"Lookup variant: {LOOKUP_VARIANT}")
print(f"Benchmark region: {BENCH_REGION}")

## Tool Adapters And Scenarios

In [ ]:
def pb_scan(partitions: int, **kwargs) -> pl.LazyFrame:
    pb.set_option("datafusion.execution.target_partitions", str(partitions))
    return pb.scan_vcf_zarr(str(VCZ_PATH), **kwargs)


def pb_open_metadata(partitions: int) -> dict[str, Any]:
    lf = pb_scan(partitions)
    schema = lf.collect_schema()
    return {"columns": len(schema.names()), "arrays": len(ARRAY_NAMES), "variants": VARIANT_COUNT}


def pb_full_core(partitions: int) -> pl.DataFrame:
    lf = pb_scan(partitions)
    return lf.select(available_core_columns(lf)).collect()


def pb_projection_min(partitions: int) -> pl.DataFrame:
    lf = pb_scan(partitions)
    cols = [column for column in ["chrom", "start"] if column in set(lf.collect_schema().names())]
    return lf.select(cols).collect()


def pb_projection_quality(partitions: int) -> pl.DataFrame:
    lf = pb_scan(partitions)
    cols = [column for column in ["chrom", "start", "qual"] if column in set(lf.collect_schema().names())]
    if len(cols) < 3:
        raise SkipBenchmark("qual column is not available")
    return lf.select(cols).collect()


def pb_info_projection(partitions: int) -> pl.DataFrame:
    if INFO_FIELD is None:
        raise SkipBenchmark("no INFO field array found")
    lf = pb_scan(partitions, info_fields=[INFO_FIELD])
    return lf.select(["chrom", "start", INFO_FIELD]).collect()


def pb_range_filter(partitions: int) -> pl.DataFrame:
    chrom, start, end = BENCH_REGION
    lf = pb_scan(partitions)
    cols = available_core_columns(lf)
    return (
        lf.filter((pl.col("chrom") == chrom) & (pl.col("start") >= start) & (pl.col("start") <= end))
        .select(cols)
        .collect()
    )


def pb_variant_lookup(partitions: int) -> pl.DataFrame:
    chrom, position = LOOKUP_VARIANT
    lf = pb_scan(partitions)
    cols = available_core_columns(lf)
    return lf.filter((pl.col("chrom") == chrom) & (pl.col("start") == position)).select(cols).collect()


def pb_sample_format(partitions: int) -> pl.DataFrame:
    if FORMAT_FIELD is None:
        raise SkipBenchmark("no FORMAT field array found")
    if not SAMPLES_FOR_QUERY:
        raise SkipBenchmark("no sample IDs available")
    chrom, start, end = BENCH_REGION
    lf = pb_scan(partitions, format_fields=[FORMAT_FIELD], samples=SAMPLES_FOR_QUERY)
    return (
        lf.filter((pl.col("chrom") == chrom) & (pl.col("start") >= start) & (pl.col("start") <= end))
        .select(["chrom", "start", "genotypes"])
        .collect()
    )


def pb_count_by_chrom(partitions: int) -> pl.DataFrame:
    return pb_scan(partitions).group_by("chrom").agg(pl.len().alias("variants")).collect()


def zarr_open_metadata() -> dict[str, Any]:
    if ZARR_GROUP is None:
        raise SkipBenchmark("zarr-python could not open the VCZ store")
    return {"arrays": len(ARRAY_NAMES), "variants": VARIANT_COUNT, "samples": len(SAMPLE_IDS)}


def zarr_full_core() -> dict[str, Any]:
    required = ["variant_contig", "variant_position", "variant_allele"]
    for name in required:
        if not has_array(name):
            raise SkipBenchmark(f"required raw array {name!r} is missing")
    contigs = read_zarr("variant_contig")
    positions = read_zarr("variant_position")
    alleles = read_zarr("variant_allele")
    if has_array("variant_length"):
        _ = read_zarr("variant_length")
    return {"rows": len(positions), "shape": str(getattr(alleles, "shape", "unknown")), "arrays": 4 if has_array("variant_length") else 3}


def zarr_projection_min() -> dict[str, Any]:
    contigs = read_zarr("variant_contig")
    positions = read_zarr("variant_position")
    return {"rows": len(positions), "shape": f"contig={getattr(contigs, 'shape', None)}, position={getattr(positions, 'shape', None)}"}


def zarr_projection_quality() -> dict[str, Any]:
    quality = read_zarr("variant_quality")
    positions = read_zarr("variant_position")
    return {"rows": len(positions), "shape": str(getattr(quality, "shape", None))}


def zarr_info_projection() -> dict[str, Any]:
    if INFO_FIELD is None:
        raise SkipBenchmark("no INFO field array found")
    values = read_zarr(f"variant_{INFO_FIELD}")
    return {"rows": len(values), "shape": str(getattr(values, "shape", None))}


def zarr_range_filter() -> dict[str, Any]:
    chrom, start, end = BENCH_REGION
    positions = decode_list(read_zarr("variant_position"))
    if has_array("variant_contig"):
        lookup = contig_lookup()
        contigs = [lookup.get(int(value), str(value)) for value in decode_list(read_zarr("variant_contig"))]
    else:
        contigs = [chrom] * len(positions)
    indexes = [idx for idx, (contig, pos) in enumerate(zip(contigs, positions)) if contig == chrom and start <= int(pos) <= end]
    if indexes and has_array("variant_allele"):
        first = min(indexes)
        last = max(indexes) + 1
        _ = read_zarr("variant_allele", slice(first, last))
    return {"rows": len(indexes), "variants": len(positions)}


def zarr_variant_lookup() -> dict[str, Any]:
    chrom, position = LOOKUP_VARIANT
    positions = decode_list(read_zarr("variant_position"))
    if has_array("variant_contig"):
        lookup = contig_lookup()
        contigs = [lookup.get(int(value), str(value)) for value in decode_list(read_zarr("variant_contig"))]
    else:
        contigs = [chrom] * len(positions)
    indexes = [idx for idx, (contig, pos) in enumerate(zip(contigs, positions)) if contig == chrom and int(pos) == int(position)]
    return {"rows": len(indexes), "variants": len(positions)}


def zarr_sample_format() -> dict[str, Any]:
    if FORMAT_FIELD is None:
        raise SkipBenchmark("no FORMAT field array found")
    name = f"call_{FORMAT_FIELD}"
    arr = zarr_array(name)
    if not SAMPLES_FOR_QUERY:
        raise SkipBenchmark("no sample IDs available")
    sample_indexes = [SAMPLE_ID_TO_INDEX[sample] for sample in SAMPLES_FOR_QUERY if sample in SAMPLE_ID_TO_INDEX]
    if not sample_indexes:
        raise SkipBenchmark("requested samples are not present")
    variant_stop = min(arr.shape[0], 1000)
    sample_start = min(sample_indexes)
    sample_stop = max(sample_indexes) + 1
    selection = (slice(0, variant_stop), slice(sample_start, sample_stop))
    if len(arr.shape) > 2:
        selection = (*selection, *[slice(None) for _ in arr.shape[2:]])
    data = arr[selection]
    return {"rows": variant_stop, "samples": sample_stop - sample_start, "shape": str(getattr(data, "shape", None))}


def zarr_count_by_chrom() -> dict[str, Any]:
    if not has_array("variant_contig"):
        raise SkipBenchmark("variant_contig is missing")
    lookup = contig_lookup()
    contigs = [lookup.get(int(value), str(value)) for value in decode_list(read_zarr("variant_contig"))]
    counts = pd.Series(contigs).value_counts()
    return {"rows": len(counts), "variants": int(counts.sum())}


def open_sgkit_dataset():
    errors = []
    if sgkit is not None and hasattr(sgkit, "load_dataset"):
        try:
            return sgkit.load_dataset(str(VCZ_PATH))
        except Exception as exc:
            errors.append(f"sgkit.load_dataset: {exc}")
    if xarray is not None and hasattr(xarray, "open_zarr"):
        try:
            return xarray.open_zarr(str(VCZ_PATH), consolidated=False)
        except Exception as exc:
            errors.append(f"xarray.open_zarr: {exc}")
    raise SkipBenchmark("; ".join(errors) or "sgkit/xarray are not installed")


def sgkit_vars(ds) -> set[str]:
    return set(getattr(ds, "data_vars", {}).keys())


def sgkit_require(ds, names: list[str]) -> None:
    available = sgkit_vars(ds)
    missing = [name for name in names if name not in available]
    if missing:
        raise SkipBenchmark(f"sgkit/xarray dataset missing variables: {missing}")


def sgkit_open_metadata() -> dict[str, Any]:
    ds = open_sgkit_dataset()
    return {"arrays": len(sgkit_vars(ds)), "shape": dict(getattr(ds, "sizes", {}))}


def sgkit_load_vars(names: list[str]) -> Any:
    ds = open_sgkit_dataset()
    sgkit_require(ds, names)
    return ds[names].load()


def sgkit_full_core() -> Any:
    return sgkit_load_vars(["variant_contig", "variant_position", "variant_allele"])


def sgkit_projection_min() -> Any:
    return sgkit_load_vars(["variant_contig", "variant_position"])


def sgkit_projection_quality() -> Any:
    return sgkit_load_vars(["variant_position", "variant_quality"])


def sgkit_info_projection() -> Any:
    if INFO_FIELD is None:
        raise SkipBenchmark("no INFO field array found")
    return sgkit_load_vars([f"variant_{INFO_FIELD}"])


def sgkit_range_filter() -> Any:
    ds = open_sgkit_dataset()
    sgkit_require(ds, ["variant_position"])
    _, start, end = BENCH_REGION
    positions = ds["variant_position"].load()
    mask = (positions >= start) & (positions <= end)
    return ds.isel(variants=mask).load()


def sgkit_variant_lookup() -> Any:
    ds = open_sgkit_dataset()
    sgkit_require(ds, ["variant_position"])
    _, position = LOOKUP_VARIANT
    positions = ds["variant_position"].load()
    mask = positions == position
    return ds.isel(variants=mask).load()


def sgkit_sample_format() -> Any:
    ds = open_sgkit_dataset()
    available = sgkit_vars(ds)
    for candidate in ["call_GT", "call_genotype", "call_DP"]:
        if candidate in available:
            data_var = candidate
            break
    else:
        raise SkipBenchmark("no compatible call field found")
    if "samples" not in ds.sizes:
        raise SkipBenchmark("dataset has no samples dimension")
    sample_stop = min(SAMPLE_COUNT, int(ds.sizes["samples"]))
    variant_stop = min(1000, int(ds.sizes.get("variants", 1000)))
    return ds[[data_var]].isel(variants=slice(0, variant_stop), samples=slice(0, sample_stop)).load()


def sgkit_count_by_chrom() -> Any:
    return sgkit_load_vars(["variant_contig"])

## Run Benchmarks

In [ ]:
POLARS_SCENARIOS: list[tuple[str, Callable[[int], Any], str]] = [
    ("open_metadata", pb_open_metadata, "DataFusion schema creation; no row materialization"),
    ("full_core_scan", pb_full_core, "Logical VCF core columns"),
    ("projection_chrom_start", pb_projection_min, "Projection pruning to chrom/start"),
    ("projection_quality", pb_projection_quality, "Projection including quality"),
    ("info_projection", pb_info_projection, "Projected INFO field"),
    ("range_filter", pb_range_filter, "Genomic range filter"),
    ("variant_lookup", pb_variant_lookup, "Exact variant lookup"),
    ("sample_format", pb_sample_format, "FORMAT field for selected samples"),
    ("count_by_chrom", pb_count_by_chrom, "Simple aggregation"),
]

FIXED_SCENARIOS: dict[str, list[tuple[str, Callable[[], Any], str]]] = {
    "zarr-python": [
        ("open_metadata", zarr_open_metadata, "Raw Zarr metadata"),
        ("full_core_scan", zarr_full_core, "Raw core arrays"),
        ("projection_chrom_start", zarr_projection_min, "Raw contig/position arrays"),
        ("projection_quality", zarr_projection_quality, "Raw quality array"),
        ("info_projection", zarr_info_projection, "Raw INFO array"),
        ("range_filter", zarr_range_filter, "Raw-array range equivalent"),
        ("variant_lookup", zarr_variant_lookup, "Raw-array exact lookup"),
        ("sample_format", zarr_sample_format, "Raw FORMAT array slice"),
        ("count_by_chrom", zarr_count_by_chrom, "Raw contig aggregation"),
    ],
    "sgkit": [
        ("open_metadata", sgkit_open_metadata, "sgkit/xarray dataset metadata"),
        ("full_core_scan", sgkit_full_core, "Dataset core variables"),
        ("projection_chrom_start", sgkit_projection_min, "Dataset contig/position variables"),
        ("projection_quality", sgkit_projection_quality, "Dataset quality variable"),
        ("info_projection", sgkit_info_projection, "Dataset INFO variable"),
        ("range_filter", sgkit_range_filter, "Dataset position filter"),
        ("variant_lookup", sgkit_variant_lookup, "Dataset exact position lookup"),
        ("sample_format", sgkit_sample_format, "Dataset call field sample slice"),
        ("count_by_chrom", sgkit_count_by_chrom, "Dataset contig variable load"),
    ],
}

rows: list[ResultRow] = []

for partition in TARGET_PARTITIONS:
    for scenario, fn, notes in POLARS_SCENARIOS:
        rows.extend(
            measure(
                "polars-bio",
                scenario,
                str(partition),
                lambda fn=fn, partition=partition: fn(partition),
                notes=notes,
            )
        )

for tool, scenarios in FIXED_SCENARIOS.items():
    for scenario, fn, notes in scenarios:
        rows.extend(measure(tool, scenario, "fixed", fn, notes=notes))

raw_results_pd = pd.DataFrame([row.__dict__ for row in rows])
raw_results_pl = pl.from_pandas(raw_results_pd)

display(raw_results_pd)

## Result Tables

In [ ]:
def summarize_results(raw: pd.DataFrame) -> pd.DataFrame:
    if raw.empty:
        return pd.DataFrame()
    ok = raw[raw["status"] == "ok"]
    if ok.empty:
        return pd.DataFrame(columns=["tool", "scenario", "partition", "median_s", "min_s", "max_s", "runs", "failures"])
    summary = (
        ok.groupby(["tool", "scenario", "partition"], dropna=False)
        .agg(
            median_s=("elapsed_s", "median"),
            min_s=("elapsed_s", "min"),
            max_s=("elapsed_s", "max"),
            runs=("elapsed_s", "count"),
        )
        .reset_index()
    )
    failures = (
        raw[raw["status"] != "ok"]
        .groupby(["tool", "scenario", "partition"], dropna=False)
        .size()
        .reset_index(name="failures")
    )
    summary = summary.merge(failures, on=["tool", "scenario", "partition"], how="left")
    summary["failures"] = summary["failures"].fillna(0).astype(int)
    return summary.sort_values(["scenario", "tool", "partition"]).reset_index(drop=True)


summary_pd = summarize_results(raw_results_pd)
summary_pl = pl.from_pandas(summary_pd) if not summary_pd.empty else pl.DataFrame()
failed_or_skipped_pd = raw_results_pd[raw_results_pd["status"] != "ok"].copy()

display(summary_pd)
if not failed_or_skipped_pd.empty:
    display(failed_or_skipped_pd[["tool", "scenario", "partition", "status", "error", "notes"]])

## Plots

In [ ]:
def plot_results(summary: pd.DataFrame) -> None:
    if summary.empty:
        print("No successful benchmark rows to plot.")
        return
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
    except Exception as exc:
        print(f"Plotting dependencies unavailable: {exc}")
        return

    plt.figure(figsize=(14, 6))
    plot_df = summary.copy()
    plot_df["tool_partition"] = plot_df["tool"] + " / " + plot_df["partition"].astype(str)
    sns.barplot(data=plot_df, x="scenario", y="median_s", hue="tool_partition")
    plt.yscale("log")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Median runtime (s, log scale)")
    plt.title("VCF Zarr benchmark scenario runtime")
    plt.tight_layout()
    plt.show()

    pb_df = summary[summary["tool"] == "polars-bio"].copy()
    if pb_df.empty:
        print("No polars-bio rows available for scalability plot.")
        return
    pb_df["partition_int"] = pb_df["partition"].astype(int)
    base = pb_df.sort_values("partition_int").groupby("scenario").first()["median_s"].rename("base_s")
    pb_df = pb_df.merge(base, on="scenario")
    pb_df["speedup"] = pb_df["base_s"] / pb_df["median_s"]

    plt.figure(figsize=(12, 6))
    sns.lineplot(data=pb_df, x="partition_int", y="speedup", hue="scenario", marker="o")
    plt.axhline(1.0, color="black", linestyle="--", linewidth=1)
    plt.xlabel("datafusion.execution.target_partitions")
    plt.ylabel("Speedup vs smallest configured partition count")
    plt.title("polars-bio VCF Zarr partition scalability")
    plt.tight_layout()
    plt.show()


plot_results(summary_pd)

## Polars Result Tables

In [ ]:
raw_results_pl

In [ ]:
summary_pl